## Extraindo séries temporais para áreas de interesse

#### Este exemplo mostra como extrair séries de evapotranspiração média em pivôs centrais selecionados

### Importando a biblioteca Google Earth Engine

In [1]:
try:
    import ee 
except:
    !pip install earthengine-api
    import ee

In [2]:
# Instalando bibliotecas secundárias do OpenET

# Installing openet-core package
# https://github.com/Open-ET/openet-core

!pip install openet-core

# Installing openet.refetgee package
# https://github.com/Open-ET/openet-refet-gee

#    !pip install openet-refet-gee

### Inicializando Google Earth Engine
#### Necessário definir o projeto_id a ser utilizado

In [3]:
project_id = 'ee-nicolevramalho' #Definir o projeto pessoal

try:
    ee.Initialize(project=project_id)
except:
    ee.Authenticate()

### Importando o modelo SIMS e definido a localização do ambiente de trabalho 

In [4]:
import geopandas as gpd
import geemap
from tqdm import tqdm

import pprint
import sys
import datetime
import pandas as pd
import numpy as np
from IPython.display import Image

sys.path.append(r"E:\Workshop_ANA_2026\openet-sims") #Altera aqui

import openet.sims as model

### Variáveis

In [5]:
# Set landsat possible collections
collections = ['LANDSAT/LT04/C02/T1_L2',
               'LANDSAT/LT05/C02/T1_L2',
               'LANDSAT/LE07/C02/T1_L2',
               'LANDSAT/LC08/C02/T1_L2',
               'LANDSAT/LC09/C02/T1_L2']

# Set ET reference dataset and parameters
meteorology_source_inst = "ECMWF/ERA5_LAND/HOURLY"
meteorology_source_daily = "projects/openet/assets/meteorology/era5land/sa/daily"
et_reference_source='projects/ee-openetbrazil/assets/meteorological/daily/era5-land/sa'
et_reference_band='eto_asce'
et_reference_factor = 1
et_reference_resample = 'nearest'
et_reference_date_type = 'daily'

# Ponto de interesse
point = ee.Geometry.Point([53.672392,-28.599571]) #Cruz Alta

# Período de análise
start_date = '2020-01-01'
end_date = '2020-12-31'

# Cobertura de nuvens
cloud_cover = 50

# Lê arquivo shapefile das áreas de interesse
aoi_file = r"E:\Workshop_ANA_2026\Pivos_CRA\Cruz_Alta.shp"
aoi_info = gpd.read_file(aoi_file)
ee_aoi = geemap.geopandas_to_ee(aoi_info)

# Tamanho da imagem para a visualização
image_size = 768

In [6]:
aoi_info.head()

,Hectares,Polo_Nome,Polo_Tipo,UGRH_Nome,UGRH_ID,MUNIC_CD,MUNIC_NOME,UF,din_class,din_classF,cod_area,geometry
0,132.356653,Uruguai,Pivô Central,Uruguai,2,4314308,Pejuçara,RS,WC,SAFRA DUPLA (INVERNO),A,"POLYGON ((234752.477 6848744.356, 234695.726 6..."
1,109.059470,Uruguai,Pivô Central,Uruguai,2,4314308,Pejuçara,RS,DC,SAFRA DUPLA,B,"POLYGON ((235609.187 6849012.16, 235651.639 68..."
2,109.866315,Uruguai,Pivô Central,Uruguai,2,4314308,Pejuçara,RS,TC,SAFRA TRIPLA,C,"POLYGON ((237180.972 6849701.974, 237166.162 6..."
3,69.558130,Uruguai,Pivô Central,Uruguai,2,4314308,Pejuçara,RS,TC,SAFRA TRIPLA,D,"POLYGON ((237029.293 6849542.266, 237060.827 6..."
4,66.492168,Uruguai,Pivô Central,Uruguai,2,4314308,Pejuçara,RS,DC,SAFRA DUPLA,E,"POLYGON ((238515.119 6848758.357, 238474.931 6..."


### Criando o objeto do modelo

In [7]:
# Gera a Coleção de Imagens SIMS
model_obj = model.Collection(
    collections=collections,
    et_reference_source=et_reference_source, 
    et_reference_band=et_reference_band,
    et_reference_factor=et_reference_factor,
    et_reference_resample=et_reference_resample,
    start_date=start_date,
    end_date=end_date,
    geometry=ee_aoi.geometry(),
    cloud_cover_max=70,
    # filter_args={},
)

# Calcula variáveis para cada imagem da coleção
overpass_coll = model_obj.overpass(variables=['ndvi','et'])

### Multiplas tabelas cada uma contendo a série temporal de um pivô selecionado

In [8]:
# Loop between flux tower sites
for i in tqdm(aoi_info.index):

    # Seleciona pivô e guarda informações
    aoi = aoi_info.loc[i:i, :]
    id = aoi['cod_area'].values[0] # TODO: Atualizar o valor "PIVO_CODE" de acordo com o .shp da área de interesse

    # Converte GeoDataFrame em ee.Feature
    ee_aoi = geemap.geopandas_to_ee(aoi_info.loc[i:i,:]).first()

    # Função para extrair valores das bandas para um pivô
    def extract_band_values(image):
        try:
            values = image.reduceRegion(
                reducer=ee.Reducer.mean(),
                geometry=ee_aoi.geometry(),
                scale=30
            )
        except:
            values = {'ndvi': np.nan, 'lst': np.nan, 'et': np.nan}
        
        return ee.Feature(None, values).set('date', image.date().format())

    # Aplicar a função em cada imagem da coleção
    time_series = overpass_coll.map(extract_band_values)

    # Exportar a série temporal como uma tabela para o Google Drive
    task = ee.batch.Export.table.toDrive(
        collection=time_series,
        folder= f'SerieTemporal_Pivos_sims_CRA',
        description=f'sims_pivo_{id}',
        selectors=['date', 'ndvi', 'et'],
        fileFormat='CSV'
    )

    task.start()

100%|██████████| 7/7 [00:05<00:00,  1.33it/s]
